In [45]:
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd

import games
from src import data as D


@dataclass
class ViterbiPossessionConfig:
    """Decodificador Viterbi de posesion L/A/D (3 estados, sin X).

    Procedencia de los valores (regla del organizador: NINGUNO se calibra con la GT del reto;
    la GT solo se usa para evaluar, nunca para seleccionar):
      - IDSSE (partidos externos, etiquetas algoritmicas): stat_threshold_dead, p_stay_dead,
        min_dead_frames -- ver evaluation/gt_free_diagnostics.py parte B
        (IDSSE: frac_parados D 0.311 vs vivo 0.046; corridas D p50 19.7s, 0% < 2s).
      - Estadisticos del propio tracking (sin etiquetas): umbrales de velocidad de balon
        (lento < 2.5 m/s, rapido > 4 m/s) y radio de control, medidas sobre el parquet.
      - Literatura / propuestas del diseno: beta=0.15, ventana de mediana movil 5 min,
        filtros morfologicos.
    """
    fps: float = 25.0
    pitch_l: float = games.GAMES[games.GAME]["pitch"][0]
    pitch_w: float = games.GAMES[games.GAME]["pitch"][1]

    # --- (a) atenuacion de la emision -------------------------------------------------------
    # Los frames a 25 Hz con EMA bidireccional NO son independientes. PROCEDENCIA: el LOMO
    # sobre IDSSE (evaluation/select_viterbi_idsse.py, spellF1 sobre etiquetas de posesion
    # externas, 7 folds) eligio beta=0.7 -- beta y p_switch quedaron casi planos en la
    # rejilla (seleccion debil, como predijo el analisis de dinamica: en el regimen de
    # "gana el ultimo control claro" el filtro morfologico manda), pero0.7 gano de forma
    # estable. La GT publica no participo.
    beta: float = 0.7

    # --- (d) posesion SOLO con control efectivo, evidencia saturada -------------------------
    control_radius_m: float = 2.0        # d_min por debajo = candidato a control
    duel_gap_m: float = 0.7              # duelo real = |d_L - d_A| < 0.7 m sin control para nadie
    w_duel_pulse: float = 1.0             # 6.3: pulso suave al equipo mas cerca tras un vuelo
                                          # si aterriza sin evidencia clara (duelo O suelo);
                                          # LOMO IDSSE: elegido en los7 folds (0.0 -> 1.0)
    flight_speed_mps: float = 8.512       # 7.0 re-escalado por cuantil a la escala ATD (part E:
                                          # CDF 0.753 en IDSSE, nuestra velocidad de balon es
                                          # 2.6x la de IDSSE por ruido de deteccion)
    contest_soft_m: float = 1.2          # escala del margen de claridad entre equipos
    w_control: float = 2.6               # techo de evidencia por control (saturado, no crece)
    ema_smooth_alpha: float = 0.35       # suavizado de d_min (sin desfase, bidireccional)
    # NOTA: se eliminaron el canal territorial, la velocidad de equipo y los conteos 3m/5m:
    # la posesion cambia solo por control efectivo del rival, no por proximidad ni formacion.

    # --- (b) canal D: el balon manda, AND de senales, entrada/salida asimetrica --------------
    ball_slow_mps: float = 4.765         # 3.0 (IDSSE: 100% de muertos <3.0) RE-ESCALADO por
                                          # cuantil a la escala ATD (part E, CDF 0.575): la
                                          # velocidad de balon propia es 2.6x la de IDSSE
    fast_ball_mps: float = 5.558          # 4.0 re-escalado por cuantil (part E, CDF 0.624)
    w_dead_ball_slow: float = 7.0         # peso balon lento (claramente mayor que jugadores:
                                          # round3 GT-free mostro 4.0 -> 7.0 mejor D-share)
    dead_players_floor: float = 0.4       # jerarquia del diseno: el balón manda, los jugadores
                                          # solo modulan (0 = producto puro hunde D cuando los
                                          # jugadores joggean durante el paron: el detector puro
                                          # sobre IDSSE solo alcanza 17.9% vs 44.9% del proveedor)
    w_dead_ball_oob: float = 5.0          # fuera de limites: concluyente por regla
    w_dead_ball_missing: float = 2.0      # sin balon Y jugadores frenados (perdida de tracking)
    w_dead_fast_ball: float = 4.0         # evidencia NEGATIVA: balon rapido = juego
    stat_threshold_dead: float = 0.273    # 0.15 (IDSSE D 0.311 vs vivo 0.046) RE-ESCALADO por
                                          # cuantil (part E, CDF 0.681): frac_stationary
                                          # propio es 2.7x la de IDSSE
    rolling_median_frames: int = 7500     # 5 min: baseline local de velocidad (mediana movil)
    slow_gate_frames: int = 50            # ~2 s: sustain para entrar en D (balon lento)
    fast_gate_frames: int = 12            # ~0.5 s: respuesta rapida para salir de D

    # --- matriz de transicion 3x3 ------------------------------------------------------------
    p_stay_play: float = 0.9974
    p_switch_play: float = 0.005          # prior L<->A ~0.005 (tasa IDSSE cruda implícita);
                                           # LOMO IDSSE lo dejo practicamente plano vs 0.008
    p_play_to_dead: float = 0.00035       # entrar en D: muy costoso, evidencia sostenida
    p_stay_dead: float = 0.9975           # prior ~16 s; IDSSE corridas D p50 19.7 s
    p_dead_to_play: float = 0.0025        # salir: mas barato, ademas lo empuja el balon rapido

    # --- (g) filtros morfologicos ------------------------------------------------------------
    min_possession_frames: int = 25       # LOMO IDSSE: elegido en los7 folds; a 40 el filtro
                                           # absorbia reclamaciones de A 5.1x mas que de L (D1)
    min_d_poss_frames: int = 50            # C3: D -> posesion <2s -> D absorbida hacia D;
                                           # IDSSE: solo5 runs/7 partidos (41 frames), nosotros
                                           # 38 runs/1006 frames -- estadistica externa
    min_dead_frames: int = 50             # 2 s: IDSSE tiene 0% de corridas D < 2 s


def _bidirectional_ema(arr: np.ndarray, alpha: float) -> np.ndarray:
    """Suavizado exponencial bidireccional sin desfase para eliminar picos de 1-2 frames."""
    n = len(arr)
    if n == 0:
        return arr
    fwd = np.empty_like(arr, dtype=np.float64)
    bwd = np.empty_like(arr, dtype=np.float64)
    fwd[0] = arr[0]
    for i in range(1, n):
        fwd[i] = alpha * arr[i] + (1.0 - alpha) * fwd[i - 1]
    bwd[-1] = arr[-1]
    for i in range(n - 2, -1, -1):
        bwd[i] = alpha * arr[i] + (1.0 - alpha) * bwd[i + 1]
    return 0.5 * (fwd + bwd)


def _roll_mean(arr: np.ndarray, window: int) -> np.ndarray:
    """Media movil CENTRADA: el procesamiento es offline, asi la entrada en D (slow_gate) no
    llega ~1s tarde ni la salida (fast_gate) ~0.25s tarde -- ambos desfasajes recortaban el
    IoU de los segmentos D."""
    return (pd.Series(arr).rolling(window, center=True, min_periods=1)
            .mean().to_numpy(dtype=np.float64))


def _ema_valid_runs(raw: np.ndarray, valid: np.ndarray, alpha: float,
                    fill: float = 50.0) -> np.ndarray:
    """EMA bidireccional SOLO dentro de tramos contiguos con balon valido.

    Un EMA global con nan->50 arrastra el 50 hacia dentro de los bordes de cada hueco de
    balon, atenuando la evidencia de control exactamente al recuperar el balon. Fuera de los
    tramos validos queda `fill` (el gate de control exige ball_ok, asi que nunca se usa).
    """
    out = np.full(len(raw), fill, dtype=np.float64)
    idx = np.flatnonzero(valid)
    if not len(idx):
        return out
    splits = np.flatnonzero(np.diff(idx) > 1) + 1
    for run in np.split(idx, splits):
        if not len(run):
            continue
        out[run] = _bidirectional_ema(np.nan_to_num(raw[run], nan=fill), alpha)
    return out


class PossessionViterbiHMM:
    STATES = ["L", "A", "D"]

    def __init__(self, cfg: ViterbiPossessionConfig | None = None):
        self.cfg = cfg or ViterbiPossessionConfig()
        self.state_to_idx = {s: i for i, s in enumerate(self.STATES)}
        self.log_T = self._build_transition_matrix()
        self.log_init = np.log(np.array([0.40, 0.40, 0.20], dtype=np.float64))

    def _build_transition_matrix(self) -> np.ndarray:
        c = self.cfg
        T = np.array(
            [
                # Desde L -> [L, A, D]
                [c.p_stay_play, c.p_switch_play, c.p_play_to_dead],
                # Desde A -> [L, A, D]
                [c.p_switch_play, c.p_stay_play, c.p_play_to_dead],
                # Desde D -> [L, A, D]
                [c.p_dead_to_play, c.p_dead_to_play, c.p_stay_dead],
            ],
            dtype=np.float64,
        )
        T = T / T.sum(axis=1, keepdims=True)
        return np.log(np.maximum(T, 1e-12))

    def _compute_log_emissions(self, df: pd.DataFrame) -> np.ndarray:
        c = self.cfg
        n = len(df)
        ball_ok = df["ball_available"].to_numpy(dtype=bool)

        # --- (f) confianza continua del balon por frame (DETECTED .85 / FOT .83 /
        # KINEMATIC .62 / NONE 0) en vez del booleano ball_ok ---
        if "ball_score" in df.columns:
            ball_conf = np.where(ball_ok, np.clip(df["ball_score"].to_numpy(dtype=np.float64), 0.0, 1.0), 0.0)
        else:
            ball_conf = ball_ok.astype(np.float64)

        bx = np.nan_to_num(df["ball_x_m"].to_numpy(dtype=np.float64), nan=0.0)
        by = np.nan_to_num(df["ball_y_m"].to_numpy(dtype=np.float64), nan=0.0)
        if "ball_speed_mps" in df.columns:
            bspd = np.nan_to_num(df["ball_speed_mps"].to_numpy(dtype=np.float64), nan=1.0)
        else:
            # velocidad SOLO entre frames con balon valido: nan->0 + gradiente fabrica picos
            # falsos en los bordes de los huecos de tracking (21% de nuestros frames sin balon)
            # que disparan el umbral 'rapido' y matan slow_sust justo en los parones.
            valid_pair = ball_ok.copy()
            valid_pair[1:] &= ball_ok[:-1]
            dx = np.zeros(n)
            dy = np.zeros(n)
            dx[1:] = bx[1:] - bx[:-1]
            dy[1:] = by[1:] - by[:-1]
            bspd = np.where(valid_pair, np.hypot(dx, dy) * c.fps, np.nan)

        # (c) origen verificado: esquina 0..pitch (solo ~1% de frames con balon fuera de margen)
        in_field = (
            (bx > -0.35) & (bx < c.pitch_l + 0.35)
            & (by > -0.35) & (by < c.pitch_w + 0.35)
        )
        oob = ball_ok & ~in_field
        # con velocidad desconocida (NaN) el balon es ni lento ni rapido: la comparacion con
        # NaN ya devuelve False en ambos lados
        slow_ball = ball_ok & np.isfinite(bspd) & (bspd < c.ball_slow_mps)
        fast_ball = in_field & np.isfinite(bspd) & (bspd > c.fast_ball_mps)

        # --- (d) control efectivo: candidato a posesion solo con balon controlado ------------
        # EMA por tramos validos (ver _ema_valid_runs): sin contaminar los bordes de hueco
        d_L = _ema_valid_runs(df["d_min_L"].to_numpy(dtype=np.float64), ball_ok, c.ema_smooth_alpha)
        d_A = _ema_valid_runs(df["d_min_A"].to_numpy(dtype=np.float64), ball_ok, c.ema_smooth_alpha)
        flight = bspd > c.flight_speed_mps
        # control claro: un equipo dentro del radio y el otro lejos, O dentro de los dos pero
        # con el primero claramente mas cerca (duelo real = |dL-dA| < duel_gap_m -> sin control
        # para nadie; IDSSE: solo 22% del tiempo vivo con el rival a >=2m, el gap lo abre)
        clear_L = (d_A >= c.control_radius_m) | ((d_A < c.control_radius_m)
                                                 & ((d_A - d_L) >= c.duel_gap_m))
        clear_A = (d_L >= c.control_radius_m) | ((d_L < c.control_radius_m)
                                                 & ((d_L - d_A) >= c.duel_gap_m))
        solo_L = ball_ok & ~flight & (d_L < c.control_radius_m) & clear_L
        solo_A = ball_ok & ~flight & (d_A < c.control_radius_m) & clear_A
        margin_L = np.clip((d_A - d_L) / c.contest_soft_m, 0.0, 1.0)
        margin_A = np.clip((d_L - d_A) / c.contest_soft_m, 0.0, 1.0)

        # --- (b) evidencia de D: el balon manda ------------------------------------------------
        # sustento: fraccion de los ultimos N frames cumpliendo (ventanas causales)
        oob_sust = np.clip((_roll_mean(oob.astype(np.float64), c.slow_gate_frames) - 0.30) / 0.70, 0.0, 1.0)
        slow_sust = np.clip((_roll_mean(slow_ball.astype(np.float64), c.slow_gate_frames) - 0.60) / 0.40, 0.0, 1.0)
        fast_sust = np.clip((_roll_mean(fast_ball.astype(np.float64), c.fast_gate_frames) - 0.50) / 0.50, 0.0, 1.0)
        missing_sust = np.clip((_roll_mean((~ball_ok).astype(np.float64), c.slow_gate_frames) - 0.70) / 0.30, 0.0, 1.0)

        # jugadores frenados respecto a su propia mediana movil de 5 min (b3): sin umbral
        # absoluto; una fase tranquila de juego no dispara D porque la baseline baja con ella
        mean_spd = 0.5 * (df["mean_speed_L"].to_numpy(dtype=np.float64)
                          + df["mean_speed_A"].to_numpy(dtype=np.float64))
        roll_med = (pd.Series(mean_spd)
                    .rolling(c.rolling_median_frames, center=True, min_periods=375)
                    .median().to_numpy(dtype=np.float64))
        # NaN-safe: los 727 frames sin jugadores visibles del final del 1er tiempo no deben
        # propagar NaN a la cadena (slow_rel = 0 => sin evidencia => persistencia)
        slow_rel = np.where(np.isfinite(mean_spd) & np.isfinite(roll_med),
                            np.clip((roll_med - mean_spd) / np.maximum(roll_med, 1.0), 0.0, 1.0),
                            0.0)
        frac_stat = 0.5 * (np.nan_to_num(df["frac_stationary_L"].to_numpy(dtype=np.float64), nan=0.0)
                           + np.nan_to_num(df["frac_stationary_A"].to_numpy(dtype=np.float64), nan=0.0))
        excess_stat = np.clip((frac_stat - c.stat_threshold_dead) / (1.0 - c.stat_threshold_dead), 0.0, 1.0)
        players_slow = np.clip(np.maximum(slow_rel, excess_stat), 0.0, 1.0)

        # producto AND de senales con piso a favor del balon: balon lento Y (jugadores
        # frenados, como modulacion). Sin balon lento el producto es cero: los jugadores
        # solos no activan D. El piso respeta la jerarquia de pesos del diseno (balon >>
        # jugadores) y evita que un paron con jugadores repositioning se escape de D.
        stop_evid = slow_sust * (c.dead_players_floor
                                 + (1.0 - c.dead_players_floor) * players_slow)
        dead_evid = np.clip(np.maximum(oob_sust, stop_evid), 0.0, 1.0)

        # (6.3) pulso suave: tras un vuelo, si el balon aterriza SIN evidencia clara para
        # nadie (duelo O suelo -- D2: 40% de los aterrizajes caen en 'loose'), el equipo mas
        # cerca recibe evidencia que decae en ~0.5 s en vez de cero para ambos
        if c.w_duel_pulse > 0.0:
            flying = (~ball_ok) | ~np.isfinite(bspd) | (bspd > c.flight_speed_mps)
            since_end = np.zeros(n, dtype=np.float64)
            cnt = 0.0
            for i in range(n):
                cnt = 0.0 if flying[i] else cnt + 1.0
                since_end[i] = cnt
            zero_zone = (~flying) & ~solo_L & ~solo_A
            decay = np.exp(-since_end / 6.0) * (since_end <= 12.0)
            closer_L = (d_L <= d_A).astype(np.float64)
            pulse_L = c.w_duel_pulse * decay * zero_zone * closer_L * (1.0 - dead_evid)
            pulse_A = c.w_duel_pulse * decay * zero_zone * (1.0 - closer_L) * (1.0 - dead_evid)
        else:
            pulse_L = np.zeros(n, dtype=np.float64)
            pulse_A = np.zeros(n, dtype=np.float64)

        # --- emisiones ------------------------------------------------------------------------
        # (d) posesion: solo con control, saturada, y cero si hay evidencia de D (balon lento
        # o fuera: quien lo lleva caminando no esta jugando). Sin control -> L = A = 0, el
        # modelo mantiene el ultimo equipo (la regla del reto).
        score_L = (c.w_control * ball_conf * solo_L.astype(np.float64)
                   * (0.55 + 0.45 * margin_L) * (1.0 - dead_evid)) + pulse_L
        score_A = (c.w_control * ball_conf * solo_A.astype(np.float64)
                   * (0.55 + 0.45 * margin_A) * (1.0 - dead_evid)) + pulse_A
        score_D = (c.w_dead_ball_oob * oob_sust
                   + c.w_dead_ball_slow * stop_evid
                   + c.w_dead_ball_missing * missing_sust * players_slow
                   - c.w_dead_fast_ball * fast_sust)

        scores = np.stack([score_L, score_A, score_D], axis=1)
        # cinturon y tiradores: ningun NaN puede llegar a la cadena Viterbi (0 = sin evidencia)
        scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
        m = np.max(scores, axis=1, keepdims=True)
        log_emit = scores - (m + np.log(np.sum(np.exp(scores - m), axis=1, keepdims=True)))
        # (a) atenuacion: la evidencia a 25 Hz correlacionada no cuenta como 25 muestras
        log_emit = c.beta * log_emit
        return log_emit

    def _viterbi_decode(self, log_emit: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        n, k = log_emit.shape
        dp = np.full((n, k), -np.inf, dtype=np.float64)
        backptr = np.zeros((n, k), dtype=np.int32)
        states = np.arange(k)

        dp[0] = self.log_init + log_emit[0]
        for t in range(1, n):
            scores = dp[t - 1][:, None] + self.log_T
            best = np.max(scores, axis=0)
            # Empates hacia la transicion 'stay' (j == estado destino). Un orden fijo de
            # indices favorece siempre al 0 (L): medido, rompia la simetria L/A en el 8.7%
            # de los frames (la paridad dp[L]==dp[A] es exacta en el 25% de los tramos sin
            # evidencia, y en el limite stay/switch el argmax eligia j=0).
            tie_self = scores[states, states] == best
            first_best = np.argmax(scores == best[None, :], axis=0)
            backptr[t] = np.where(tie_self, states, first_best)
            dp[t] = scores[backptr[t], states] + log_emit[t]

        path = np.zeros(n, dtype=np.int32)
        path[-1] = int(np.argmax(dp[-1]))
        for t in range(n - 2, -1, -1):
            path[t] = backptr[t + 1, path[t + 1]]

        return path, np.exp(log_emit)

    def _absorb_short_segments(self, path_idx: np.ndarray) -> np.ndarray:
        """Elimina rebotes L<->A cortos (g) y micro-parones D falsos."""
        if len(path_idx) < 3:
            return path_idx

        idx_L = self.state_to_idx["L"]
        idx_A = self.state_to_idx["A"]
        idx_D = self.state_to_idx["D"]
        out = path_idx.copy()

        for _ in range(3):
            runs = []
            start = 0
            for i in range(1, len(out)):
                if out[i] != out[start]:
                    runs.append([start, i, int(out[start])])
                    start = i
            runs.append([start, len(out), int(out[start])])

            changed = False
            for r_i in range(1, len(runs) - 1):
                s, e, st = runs[r_i]
                dur = e - s
                prev_st = runs[r_i - 1][2]
                next_st = runs[r_i + 1][2]

                if st == idx_D and dur < self.cfg.min_dead_frames:
                    if prev_st == next_st:
                        out[s:e] = prev_st
                    else:
                        mid = s + dur // 2
                        out[s:mid] = prev_st
                        out[mid:e] = next_st
                    changed = True

                elif st in (idx_L, idx_A):
                    if (dur < self.cfg.min_possession_frames
                            and prev_st == next_st and prev_st in (idx_L, idx_A)):
                        out[s:e] = prev_st
                        changed = True
                    elif (dur < self.cfg.min_d_poss_frames
                            and prev_st == idx_D and next_st == idx_D):
                        # C3: D -> posesion corta -> D parte parones largos. En IDSSE este
                        # patron casi no existe (5 runs <2s en7 partidos =41 frames) frente a
                        # 38 runs/1006 frames en nuestro partido: absorbido hacia D, regla
                        # justificada por estadistica externa, no por GT.
                        out[s:e] = idx_D
                        changed = True

            if not changed:
                break
        return out

    def predict(self, df_features: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
        out_parts = []
        for _, df_p in df_features.groupby("period", sort=False):
            df_p = df_p.sort_values("video_frame").reset_index(drop=True)
            log_emit = self._compute_log_emissions(df_p)
            path_idx, probs = self._viterbi_decode(log_emit)
            path_idx = self._absorb_short_segments(path_idx)

            res = df_p[["video_frame", "time_s", "period", "ball_available"]].copy()
            res["pred_state"] = [self.STATES[i] for i in path_idx]
            for s_idx, s_name in enumerate(self.STATES):
                res[f"prob_{s_name}"] = probs[:, s_idx]
            out_parts.append(res)

        df_viterbi = pd.concat(out_parts, ignore_index=True)
        n_tot = len(df_viterbi)
        n_switches = int((df_viterbi["pred_state"] != df_viterbi["pred_state"].shift(1)).sum() - 1)

        st_arr = df_viterbi["pred_state"].to_numpy()
        runs = []
        s0 = 0
        for i in range(1, len(st_arr)):
            if st_arr[i] != st_arr[s0]:
                runs.append((st_arr[s0], i - s0))
                s0 = i
        runs.append((st_arr[s0], len(st_arr) - s0))
        d_runs = [d for s, d in runs if s == "D"]

        summary = {
            "total_frames": n_tot,
            "pct_L": f"{100.0 * (df_viterbi['pred_state'] == 'L').mean():.2f}%",
            "pct_A": f"{100.0 * (df_viterbi['pred_state'] == 'A').mean():.2f}%",
            "pct_D": f"{100.0 * (df_viterbi['pred_state'] == 'D').mean():.2f}%",
            "state_switches": n_switches,
            "avg_segment_s": round((n_tot / max(1, n_switches + 1)) / self.cfg.fps, 2),
            "dead_segments_count": len(d_runs),
            "min_dead_segment_s": round(min(d_runs) / self.cfg.fps, 2) if d_runs else 0.0,
        }
        return df_viterbi, summary


In [43]:

# ==============================================================================
# EJECUCIÓN DE CELDA 1: Cargar features procesados -> Decodificar con Viterbi
# ==============================================================================
df_features = pd.read_parquet(PARQUET_PATH)

viterbi_cfg = ViterbiPossessionConfig()
hmm_model = PossessionViterbiHMM(viterbi_cfg)
df_viterbi, viterbi_summary = hmm_model.predict(df_features)

print("=== RESUMEN DECODIFICACIÓN VITERBI (L / A / D) ===")
for k, v in viterbi_summary.items():
    print(f"  {k:24s}: {v}")

print("\nPrimeras predicciones:")
print(df_viterbi.head(10).to_string(index=False))

=== RESUMEN DECODIFICACIÓN VITERBI (L / A / D) ===
  total_frames            : 158857
  pct_L                   : 37.02%
  pct_A                   : 20.96%
  pct_D                   : 42.01%
  state_switches          : 662
  avg_segment_s           : 9.58
  dead_segments_count     : 203
  min_dead_segment_s      : 2.0

Primeras predicciones:
 video_frame  time_s     period  ball_available pred_state   prob_L   prob_A   prob_D
       11185  447.40 first_half           False          A 0.564470 0.564470 0.221972
       11186  447.44 first_half            True          A 0.193867 0.919303 0.058392
       11187  447.48 first_half            True          A 0.587995 0.587995 0.144998
       11188  447.52 first_half            True          A 0.593250 0.593250 0.125218
       11189  447.56 first_half            True          A 0.596751 0.596751 0.111219
       11190  447.60 first_half            True          A 0.599217 0.599217 0.100867
       11191  447.64 first_half            True       

In [44]:
diag_stats = df_viterbi.merge(
    df_features[
        [
            "video_frame",
            "mean_speed_L",
            "mean_speed_A",
            "frac_stationary_L",
            "frac_stationary_A",
        ]
    ],
    on="video_frame",
)
diag_stats["mean_spd"] = 0.5 * (diag_stats["mean_speed_L"] + diag_stats["mean_speed_A"])
diag_stats["frac_stat"] = 0.5 * (diag_stats["frac_stationary_L"] + diag_stats["frac_stationary_A"])

# Intentar cargar Ground Truth directamente desde src.data
gt_loaded = False
if hasattr(D, "load_labels"):
    try:
        gt_raw = D.load_labels()
        if gt_raw is not None and len(gt_raw) > 0:
            gt_df = gt_raw.copy()
            if "video_frame" not in gt_df.columns:
                gt_df["video_frame"] = (gt_df["frame_id"] - 1) if "frame_id" in gt_df.columns else np.arange(len(gt_df))
            lbl_col = next((c for c in ["label", "state", "possession", "target", "y"] if c in gt_df.columns), None)
            if lbl_col is not None:
                gt_df["gt_state"] = gt_df[lbl_col].astype(str).str.strip().str.upper()
                diag_stats = diag_stats.merge(gt_df[["video_frame", "gt_state"]], on="video_frame", how="inner")
                diag_stats = diag_stats[diag_stats["gt_state"].isin(["L", "A", "D"])]
                gt_loaded = len(diag_stats) > 0
    except Exception:
        pass

if gt_loaded:
    print("=== ESTADÍSTICAS POR ESTADO REAL (GROUND TRUTH) ===")
    print(diag_stats.groupby("gt_state")[["mean_spd", "frac_stat", "ball_available"]].agg(["mean", "median"]))
else:
    print("=== SIN GROUND TRUTH: ESTADÍSTICAS POR PREDICCIÓN VITERBI ===")
    print(diag_stats.groupby("pred_state")[["mean_spd", "frac_stat", "ball_available"]].agg(["mean", "median"]))

=== SIN GROUND TRUTH: ESTADÍSTICAS POR PREDICCIÓN VITERBI ===
            mean_spd          frac_stat          ball_available       
                mean   median      mean   median           mean median
pred_state                                                            
A           2.261710  2.10000  0.137443  0.09545       0.856864    1.0
D           1.050715  0.97025  0.377493  0.34090       0.686490    1.0
L           2.108937  1.94950  0.141526  0.09545       0.859076    1.0


In [27]:
import os
from pathlib import Path
from typing import Optional

import cv2
import numpy as np
import pandas as pd


def export_tactical_window_video(
    df_full: pd.DataFrame,
    slot_team: np.ndarray,
    output_path: Optional[Path] = None,
    seed: int = 42,
    duration_s: float = 20.0,
    fps: float = 25.0,
    pitch_l: float = 105.3,
    pitch_w: float = 68.0,
    scale_px_per_m: int = 12,
    period_filter: Optional[str] = None,  # None (cualquiera), "first_half" o "second_half"
    df_viterbi: Optional[pd.DataFrame] = None,
) -> Path:
    """
    Renderiza un video MP4 (OpenCV) de una ventana continua de `duration_s` segundos
    elegida reproducibly mediante `seed`, graficando posiciones, velocidades, centroides,
    líneas defensivas, radios de presión (3/5/10 m) y estado del balón / Viterbi.
    """
    # 1. Seleccionar ventana reproducible dentro de una misma mitad (sin cruzar el descanso)
    rng = np.random.default_rng(seed)
    n_win = int(round(duration_s * fps))

    periods = [period_filter] if period_filter in ("first_half", "second_half") else ["first_half", "second_half"]
    valid_starts = []
    for p in periods:
        p_idx = np.where((df_full["period"] == p).to_numpy())[0]
        if len(p_idx) > n_win:
            # Guardamos pares (start_idx, end_idx) continuos dentro de la mitad
            max_start = p_idx[-1] - n_win
            valid_starts.extend(range(int(p_idx[0]), int(max_start) + 1))

    if not valid_starts:
        raise ValueError("No hay suficientes frames continuos para la duración solicitada.")

    start_idx = int(rng.choice(valid_starts))
    win = df_full.iloc[start_idx : start_idx + n_win].reset_index(drop=True)

    # Unir predicción de Viterbi si se proporciona
    has_viterbi = False
    if df_viterbi is not None and "pred_state" in df_viterbi.columns:
        win = win.merge(df_viterbi[["video_frame", "pred_state"]], on="video_frame", how="left")
        has_viterbi = True
    elif "pred_state" in win.columns:
        has_viterbi = True

    # 2. Configurar dimensiones del lienzo (cancha + márgenes + HUD superior/inferior)
    margin_x = 60
    hud_top = 85
    hud_bot = 65
    pitch_px_w = int(round(pitch_l * scale_px_per_m))
    pitch_px_h = int(round(pitch_w * scale_px_per_m))
    canvas_w = pitch_px_w + 2 * margin_x
    canvas_h = pitch_px_h + hud_top + hud_bot

    def m2px(x_m: float, y_m: float):
        px = int(round(margin_x + x_m * scale_px_per_m))
        py = int(round(hud_top + y_m * scale_px_per_m))
        return px, py

    # Pre-dibujar plantilla estática del césped y líneas reglamentarias
    base_pitch = np.full((canvas_h, canvas_w, 3), (28, 32, 36), dtype=np.uint8)
    # Césped
    cv2.rectangle(base_pitch, (margin_x, hud_top), (margin_x + pitch_px_w, hud_top + pitch_px_h), (34, 115, 52), -1)
    # Franjas alternas sutiles
    n_stripes = 14
    stripe_w = pitch_px_w / n_stripes
    for k in range(0, n_stripes, 2):
        x0 = int(round(margin_x + k * stripe_w))
        x1 = int(round(margin_x + (k + 1) * stripe_w))
        cv2.rectangle(base_pitch, (x0, hud_top), (x1, hud_top + pitch_px_h), (39, 126, 58), -1)

    line_col = (235, 240, 235)
    # Perímetro y medio campo
    cv2.rectangle(base_pitch, (margin_x, hud_top), (margin_x + pitch_px_w, hud_top + pitch_px_h), line_col, 2)
    mid_px, mid_py = m2px(pitch_l / 2.0, pitch_w / 2.0)
    cv2.line(base_pitch, (mid_px, hud_top), (mid_px, hud_top + pitch_px_h), line_col, 2)
    cv2.circle(base_pitch, (mid_px, mid_py), int(round(9.15 * scale_px_per_m)), line_col, 2)
    cv2.circle(base_pitch, (mid_px, mid_py), 4, line_col, -1)

    # Áreas grandes (16.5m x 40.32m) y áreas pequeñas (5.5m x 18.32m)
    for side in (0, 1):
        x_goal = 0.0 if side == 0 else pitch_l
        sgn = 1.0 if side == 0 else -1.0
        # Área grande
        p1 = m2px(x_goal, (pitch_w - 40.32) / 2.0)
        p2 = m2px(x_goal + sgn * 16.5, (pitch_w + 40.32) / 2.0)
        cv2.rectangle(base_pitch, (min(p1[0], p2[0]), p1[1]), (max(p1[0], p2[0]), p2[1]), line_col, 2)
        # Área chica
        q1 = m2px(x_goal, (pitch_w - 18.32) / 2.0)
        q2 = m2px(x_goal + sgn * 5.5, (pitch_w + 18.32) / 2.0)
        cv2.rectangle(base_pitch, (min(q1[0], q2[0]), q1[1]), (max(q1[0], q2[0]), q2[1]), line_col, 2)
        # Punto penal (11m)
        pen = m2px(x_goal + sgn * 11.0, pitch_w / 2.0)
        cv2.circle(base_pitch, pen, 4, line_col, -1)

    # Indicadores de dirección de ataque estandarizada (Paso 2)
    cv2.putText(base_pitch, "ATAQUE LOCAL (L) >>>", (margin_x + 15, hud_top - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.52, (255, 190, 90), 2, cv2.LINE_AA)
    cv2.putText(base_pitch, "<<< ATAQUE AWAY (A)", (margin_x + pitch_px_w - 225, hud_top - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.52, (110, 120, 255), 2, cv2.LINE_AA)

    # 3. Configurar salida MP4
    if output_path is None:
        out_dir = Path(os.environ.get("MTAH_DATA_DIR", ".")) / "processed" / "videos"
        out_dir.mkdir(parents=True, exist_ok=True)
        vf_start = int(win["video_frame"].iloc[0])
        vf_end = int(win["video_frame"].iloc[-1])
        output_path = out_dir / f"tactical_seed{seed}_{win['period'].iloc[0]}_vf{vf_start}_{vf_end}.mp4"

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (canvas_w, canvas_h))

    n_slots = len(slot_team)
    col_L = (235, 135, 35)   # Azul BGR para Local ('L')
    col_A = (60, 65, 235)    # Rojo BGR para Away ('A')

    # 4. Renderizado frame a frame
    for i in range(len(win)):
        row = win.iloc[i]
        frame = base_pitch.copy()

        # 4a. Líneas defensivas (Paso 7)
        if np.isfinite(row["def_line_x_L"]):
            dl_x, _ = m2px(row["def_line_x_L"], 0)
            for y_dash in range(hud_top, hud_top + pitch_px_h, 16):
                cv2.line(frame, (dl_x, y_dash), (dl_x, min(y_dash + 8, hud_top + pitch_px_h)), col_L, 2)
        if np.isfinite(row["def_line_x_A"]):
            da_x, _ = m2px(row["def_line_x_A"], 0)
            for y_dash in range(hud_top, hud_top + pitch_px_h, 16):
                cv2.line(frame, (da_x, y_dash), (da_x, min(y_dash + 8, hud_top + pitch_px_h)), col_A, 2)

        # 4b. Centroides de equipo (Paso 7)
        if np.isfinite(row["centroid_x_L"]) and np.isfinite(row["centroid_y_L"]):
            cL = m2px(row["centroid_x_L"], row["centroid_y_L"])
            cv2.drawMarker(frame, cL, col_L, cv2.MARKER_TILTED_CROSS, 18, 3)
        if np.isfinite(row["centroid_x_A"]) and np.isfinite(row["centroid_y_A"]):
            cA = m2px(row["centroid_x_A"], row["centroid_y_A"])
            cv2.drawMarker(frame, cA, col_A, cv2.MARKER_TILTED_CROSS, 18, 3)

        # 4c. Radios de presión del balón (3m, 5m, 10m - Paso 8)
        ball_ok = bool(row["ball_available"]) and np.isfinite(row["ball_x_m"]) and np.isfinite(row["ball_y_m"])
        b_px = None
        if ball_ok:
            b_px = m2px(row["ball_x_m"], row["ball_y_m"])
            for r_m, alpha_col in [(10.0, (170, 210, 170)), (5.0, (140, 235, 235)), (3.0, (90, 255, 255))]:
                cv2.circle(frame, b_px, int(round(r_m * scale_px_per_m)), alpha_col, 1, cv2.LINE_AA)

        near_L = int(row["nearest_slot_L"]) if ball_ok and np.isfinite(row["nearest_slot_L"]) else -1
        near_A = int(row["nearest_slot_A"]) if ball_ok and np.isfinite(row["nearest_slot_A"]) else -1

        # 4d. Jugadores (posiciones limpias + vector de velocidad Paso 5)
        for s in range(1, n_slots + 1):
            px_m = row[f"{s}_x_m"]
            py_m = row[f"{s}_y_m"]
            if not (np.isfinite(px_m) and np.isfinite(py_m)):
                continue

            pt = m2px(px_m, py_m)
            is_home = slot_team[s - 1] == 0
            p_col = col_L if is_home else col_A

            # Vector de velocidad (escala 0.45 s de proyección)
            vx, vy = row[f"{s}_vx"], row[f"{s}_vy"]
            if np.isfinite(vx) and np.isfinite(vy):
                pt_vel = m2px(px_m + 0.45 * vx, py_m + 0.45 * vy)
                cv2.arrowedLine(frame, pt, pt_vel, (245, 245, 245), 2, tipLength=0.28)

            # Línea hacia el balón si es el más cercano de su equipo
            if ball_ok and (s == near_L or s == near_A):
                cv2.line(frame, pt, b_px, p_col, 2, cv2.LINE_AA)
                cv2.circle(frame, pt, 15, (0, 255, 255), 2, cv2.LINE_AA)

            cv2.circle(frame, pt, 10, p_col, -1, cv2.LINE_AA)
            cv2.circle(frame, pt, 10, (255, 255, 255), 1, cv2.LINE_AA)
            cv2.putText(frame, str(s), (pt[0] - 8, pt[1] + 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.36, (255, 255, 255), 1, cv2.LINE_AA)

        # 4e. Balón
        if ball_ok:
            cv2.circle(frame, b_px, 7, (0, 230, 255), -1, cv2.LINE_AA)
            cv2.circle(frame, b_px, 7, (0, 0, 0), 2, cv2.LINE_AA)

        # 4f. HUD Superior: Tiempo, estado del balón y Viterbi
        mins = int(row["time_s"] // 60)
        secs = row["time_s"] % 60.0
        hud_line1 = f"Seed={seed} | {row['period']} | vf={int(row['video_frame'])} | t={mins:02d}:{secs:05.2f}"
        cv2.putText(frame, hud_line1, (margin_x, 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.62, (240, 240, 240), 2, cv2.LINE_AA)

        status_txt = "BALON: OK (Paso 8+7)" if ball_ok else "BALON: AUSENTE -> FALLBACK COLECTIVO (Paso 7)"
        status_col = (100, 235, 120) if ball_ok else (70, 175, 255)
        if bool(row["low_conf_frame"]):
            status_txt += " | LOW_CONF (<8 jug.)"
            status_col = (80, 130, 255)

        cv2.putText(frame, status_txt, (margin_x, 54),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.52, status_col, 1, cv2.LINE_AA)

        if has_viterbi:
            st = str(row["pred_state"])
            st_map = {
                "L": ("VITERBI: LOCAL (L)", col_L),
                "A": ("VITERBI: AWAY (A)", col_A),
                "D": ("VITERBI: DEAD (D)", (200, 200, 200)),
                "X": ("VITERBI: UNKNOWN (X)", (0, 215, 255)),
            }
            lbl, lbl_col = st_map.get(st, (f"VITERBI: {st}", (255, 255, 255)))
            cv2.putText(frame, lbl, (canvas_w - margin_x - 270, 38),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.72, lbl_col, 2, cv2.LINE_AA)

        # 4g. HUD Inferior: Telemetría numérica (Pasos 6, 7 y 8)
        vis_txt = f"Visibles L/A: {int(row['n_visible_L'])}/{int(row['n_visible_A'])}"
        col_txt = (
            f"VelMedia L/A: {row['mean_speed_L']:.2f}/{row['mean_speed_A']:.2f} m/s | "
            f"Parados L/A: {row['frac_stationary_L']*100:.0f}%/{row['frac_stationary_A']*100:.0f}% | "
            f"DistCentroides: {row['centroid_dist_euclid']:.1f} m"
        )
        if ball_ok:
            ball_txt = (
                f"d_min L/A: {row['d_min_L']:.2f}/{row['d_min_A']:.2f} m (diff={row['diff_d_min_LA']:+.2f}) | "
                f"CloseSpd L/A: {row['closing_speed_L']:+.1f}/{row['closing_speed_A']:+.1f} m/s | "
                f"R3m L/A: {int(row['count_3m_L'])}/{int(row['count_3m_A'])}"
            )
        else:
            ball_txt = "d_min L/A: NaN | CloseSpd: NaN | R3m: NaN"

        cv2.putText(frame, f"{vis_txt}  ||  {col_txt}", (margin_x, canvas_h - 38),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.46, (215, 225, 230), 1, cv2.LINE_AA)
        cv2.putText(frame, ball_txt, (margin_x, canvas_h - 14),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.46, (180, 230, 255), 1, cv2.LINE_AA)

        writer.write(frame)

    writer.release()
    print(f"Video exportado correctamente ({duration_s:.1f}s @ {fps} FPS):")
    print(f"  Ruta  : {output_path}")
    print(f"  Rango : video_frame {int(win['video_frame'].iloc[0])} -> {int(win['video_frame'].iloc[-1])} ({win['period'].iloc[0]})")
    return output_path


# ==============================================================================
# EJECUCIÓN DE EJEMPLO (Cambia `seed` para inspeccionar distintas jugadas)
# ==============================================================================
# Si estás en el notebook de Viterbi y cargaste df_features desde el parquet:
# df_full = pd.read_parquet(PARQUET_PATH)
# slot_team = np.where(pd.read_csv(PROCESSED_DIR / f"{games.GAME}_slot_teams_LA.csv")["team"] == "L", 0, 1)

video_file = export_tactical_window_video(
    df_full = pd.read_parquet(PARQUET_PATH),
    slot_team=np.where(pd.read_csv(PROCESSED_DIR / f"{games.GAME}_slot_teams_LA.csv")["team"] == "L", 0, 1),
    seed=42,             # Cambia este número para saltar a otra jugada aleatoria
    duration_s=20.0,     # Duración del clip en segundos
    period_filter=None,  # Opcional: "first_half" o "second_half"
    df_viterbi=df_viterbi if "df_viterbi" in globals() else None,
)

Video exportado correctamente (20.0s @ 25.0 FPS):
  Ruta  : D:\Proyectos\Barcelona project local\data\processed\videos\tactical_seed42_first_half_vf25273_25772.mp4
  Rango : video_frame 25273 -> 25772 (first_half)


In [42]:
def _extract_frame_homography(H_df: pd.DataFrame, target_vf: int, h_maps_image_to_pitch: bool = False) -> Optional[np.ndarray]:
    """Extrae la matriz 3x3 de homografía correspondiente al video_frame (frame_id = target_vf + 1)."""
    if "video_frame" in H_df.columns:
        row_h = H_df[H_df["video_frame"] == target_vf]
    elif "frame_id" in H_df.columns:
        row_h = H_df[H_df["frame_id"] == (target_vf + 1)]
    else:
        row_h = H_df.iloc[[target_vf]] if 0 <= target_vf < len(H_df) else pd.DataFrame()

    if len(row_h) == 0:
        return None

    # Excluir columnas de índice/tiempo y tomar los 9 valores de la matriz 3x3
    meta_cols = {"frame_id", "video_frame", "time_s", "period", "frame"}
    num_cols = [c for c in row_h.columns if c not in meta_cols]
    vals = row_h[num_cols].iloc[0].to_numpy(dtype=np.float64)

    if len(vals) < 9 or not np.isfinite(vals[:9]).all():
        return None

    H_mat = vals[:9].reshape(3, 3)
    if h_maps_image_to_pitch:
        try:
            H_mat = np.linalg.inv(H_mat)
        except np.linalg.LinAlgError:
            return None
    return H_mat


def export_dual_view_tactical_video(
    df_full: pd.DataFrame,
    slot_team: np.ndarray,
    atd_summary: Optional[dict] = None,
    video_path: Optional[Path] = None,
    H_df: Optional[pd.DataFrame] = None,
    h_maps_image_to_pitch: bool = False,       # Pon True si H mapea píxeles -> campo
    pitch_coords_are_normalized: bool = True,  # En ATD/homografía del reto suele ser [0, 1] x [0, 1]
    output_path: Optional[Path] = None,
    seed: int = 42,
    duration_s: float = 20.0,
    fps: float = 25.0,
    pitch_l: float = 105.3,
    pitch_w: float = 68.0,
    period_filter: Optional[str] = None,
    df_viterbi: Optional[pd.DataFrame] = None,
) -> Path:
    auto_vid, auto_H = _resolve_video_and_homography()
    video_path = Path(video_path) if video_path is not None else auto_vid
    H_source = H_df if H_df is not None else auto_H

    if video_path is None or not video_path.exists():
        raise FileNotFoundError(f"No se encontró el video del partido: {video_path}")
    if H_source is None:
        raise ValueError("No se encontró la tabla de homografía.")

    is_dynamic_h = isinstance(H_source, pd.DataFrame)
    static_H = None
    if not is_dynamic_h:
        static_H = np.asarray(H_source, dtype=np.float64).reshape(3, 3)
        if h_maps_image_to_pitch:
            static_H = np.linalg.inv(static_H)

    # 1. Seleccionar ventana reproducible mediante `seed`
    rng = np.random.default_rng(seed)
    n_win = int(round(duration_s * fps))
    periods = [period_filter] if period_filter in ("first_half", "second_half") else ["first_half", "second_half"]

    valid_starts = []
    for p in periods:
        p_idx = np.where((df_full["period"] == p).to_numpy())[0]
        if len(p_idx) > n_win:
            valid_starts.extend(range(int(p_idx[0]), int(p_idx[-1] - n_win) + 1))

    start_idx = int(rng.choice(valid_starts))
    win = df_full.iloc[start_idx : start_idx + n_win].reset_index(drop=True)

    has_viterbi = False
    if df_viterbi is not None and "pred_state" in df_viterbi.columns:
        win = win.merge(df_viterbi[["video_frame", "pred_state"]], on="video_frame", how="left")
        has_viterbi = True
    elif "pred_state" in win.columns:
        has_viterbi = True

    # 2. Abrir captura de video y configurar dimensiones del lienzo doble
    cap = cv2.VideoCapture(str(video_path))
    vid_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    vid_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    hud_top = 80
    hud_bot = 60
    margin_2d = 36
    avail_h_2d = vid_h - hud_top - hud_bot
    scale_2d = avail_h_2d / pitch_w
    pitch_px_w = int(round(pitch_l * scale_2d))
    pitch_px_h = int(round(pitch_w * scale_2d))
    right_w = pitch_px_w + 2 * margin_2d
    canvas_w = vid_w + right_w
    canvas_h = vid_h

    home_dir_fh = atd_summary.get("home_first_half_dir", 1) if atd_summary is not None else 1

    def canonical_to_raw_pitch(x_can: float, y_can: float, is_second_half: bool):
        xr, yr = x_can, y_can
        if home_dir_fh == -1:
            xr = pitch_l - xr
            yr = pitch_w - yr
        if is_second_half:
            xr = pitch_l - xr
            yr = pitch_w - yr
        return xr, yr

    def pitch_to_video_px(x_can: float, y_can: float, is_second_half: bool, H_mat: Optional[np.ndarray]):
        if H_mat is None:
            return None
        xr, yr = canonical_to_raw_pitch(x_can, y_can, is_second_half)
        if pitch_coords_are_normalized:
            xr /= pitch_l
            yr /= pitch_w
        vec = np.array([xr, yr, 1.0], dtype=np.float64)
        proj = H_mat @ vec
        if abs(proj[2]) < 1e-8:
            return None
        u = int(round(proj[0] / proj[2]))
        v = int(round(proj[1] / proj[2]))
        return (u, v)

    def pitch_to_2d_px(x_can: float, y_can: float):
        px = int(round(vid_w + margin_2d + x_can * scale_2d))
        py = int(round(hud_top + y_can * scale_2d))
        return (px, py)

    # Plantilla estática del panel derecho (pizarra 2D)
    right_template = np.full((canvas_h, right_w, 3), (26, 30, 34), dtype=np.uint8)
    x0_2d, y0_2d = margin_2d, hud_top
    cv2.rectangle(right_template, (x0_2d, y0_2d), (x0_2d + pitch_px_w, y0_2d + pitch_px_h), (34, 115, 52), -1)
    stripe_w = pitch_px_w / 14.0
    for k in range(0, 14, 2):
        cv2.rectangle(
            right_template,
            (int(round(x0_2d + k * stripe_w)), y0_2d),
            (int(round(x0_2d + (k + 1) * stripe_w)), y0_2d + pitch_px_h),
            (39, 126, 58),
            -1,
        )

    line_col = (235, 240, 235)
    cv2.rectangle(right_template, (x0_2d, y0_2d), (x0_2d + pitch_px_w, y0_2d + pitch_px_h), line_col, 2)
    mid_x_2d = x0_2d + pitch_px_w // 2
    mid_y_2d = y0_2d + pitch_px_h // 2
    cv2.line(right_template, (mid_x_2d, y0_2d), (mid_x_2d, y0_2d + pitch_px_h), line_col, 2)
    cv2.circle(right_template, (mid_x_2d, mid_y_2d), int(round(9.15 * scale_2d)), line_col, 2)

    for side in (0, 1):
        xg = 0.0 if side == 0 else pitch_l
        sgn = 1.0 if side == 0 else -1.0
        p1 = (int(round(x0_2d + xg * scale_2d)), int(round(y0_2d + ((pitch_w - 40.32) / 2.0) * scale_2d)))
        p2 = (int(round(x0_2d + (xg + sgn * 16.5) * scale_2d)), int(round(y0_2d + ((pitch_w + 40.32) / 2.0) * scale_2d)))
        cv2.rectangle(right_template, (min(p1[0], p2[0]), p1[1]), (max(p1[0], p2[0]), p2[1]), line_col, 2)

        q1 = (int(round(x0_2d + xg * scale_2d)), int(round(y0_2d + ((pitch_w - 18.32) / 2.0) * scale_2d)))
        q2 = (int(round(x0_2d + (xg + sgn * 5.5) * scale_2d)), int(round(y0_2d + ((pitch_w + 18.32) / 2.0) * scale_2d)))
        cv2.rectangle(right_template, (min(q1[0], q2[0]), q1[1]), (max(q1[0], q2[0]), q2[1]), line_col, 2)

    cv2.putText(right_template, "ATAQUE LOCAL (L) >>>", (x0_2d + 8, y0_2d - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 190, 90), 2, cv2.LINE_AA)
    cv2.putText(right_template, "<<< ATAQUE AWAY (A)", (x0_2d + pitch_px_w - 205, y0_2d - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (110, 120, 255), 2, cv2.LINE_AA)

    if output_path is None:
        out_dir = Path(os.environ.get("MTAH_DATA_DIR", ".")) / "processed" / "videos"
        out_dir.mkdir(parents=True, exist_ok=True)
        vf0 = int(win["video_frame"].iloc[0])
        vf1 = int(win["video_frame"].iloc[-1])
        output_path = out_dir / f"dual_view_seed{seed}_{win['period'].iloc[0]}_vf{vf0}_{vf1}.mp4"

    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (canvas_w, canvas_h))

    first_vf = int(win["video_frame"].iloc[0])
    cap.set(cv2.CAP_PROP_POS_FRAMES, first_vf)

    n_slots = len(slot_team)
    col_L = (235, 135, 35)
    col_A = (60, 65, 235)
    theta_circle = np.linspace(0, 2 * np.pi, 48)

    for i in range(len(win)):
        row = win.iloc[i]
        target_vf = int(row["video_frame"])
        H_mat = _extract_frame_homography(H_source, target_vf, h_maps_image_to_pitch) if is_dynamic_h else static_H

        cur_pos = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
        if cur_pos != target_vf:
            cap.set(cv2.CAP_PROP_POS_FRAMES, target_vf)
        ret, vid_frame = cap.read()
        if not ret:
            vid_frame = np.zeros((vid_h, vid_w, 3), dtype=np.uint8)

        canvas = np.hstack([vid_frame, right_template.copy()])
        is_sh = row["period"] == "second_half"
        ball_ok = bool(row["ball_available"]) and np.isfinite(row["ball_x_m"]) and np.isfinite(row["ball_y_m"])

        # 4a. Líneas defensivas en ambas vistas
        for def_col_name, team_col in [("def_line_x_L", col_L), ("def_line_x_A", col_A)]:
            dx = row[def_col_name]
            if np.isfinite(dx):
                p_top_2d = pitch_to_2d_px(dx, 0.0)
                p_bot_2d = pitch_to_2d_px(dx, pitch_w)
                for y_d in range(p_top_2d[1], p_bot_2d[1], 14):
                    cv2.line(canvas, (p_top_2d[0], y_d), (p_top_2d[0], min(y_d + 7, p_bot_2d[1])), team_col, 2)
                pts_3d = [pitch_to_video_px(dx, yv, is_sh, H_mat) for yv in np.linspace(0.0, pitch_w, 25)]
                pts_3d = [p for p in pts_3d if p is not None]
                for k in range(0, len(pts_3d) - 1, 2):
                    cv2.line(canvas, pts_3d[k], pts_3d[k + 1], team_col, 2, cv2.LINE_AA)

        # 4b. Centroides de equipo
        for cx_name, cy_name, team_col in [("centroid_x_L", "centroid_y_L", col_L), ("centroid_x_A", "centroid_y_A", col_A)]:
            cx_v, cy_v = row[cx_name], row[cy_name]
            if np.isfinite(cx_v) and np.isfinite(cy_v):
                cv2.drawMarker(canvas, pitch_to_2d_px(cx_v, cy_v), team_col, cv2.MARKER_TILTED_CROSS, 16, 3)
                p3 = pitch_to_video_px(cx_v, cy_v, is_sh, H_mat)
                if p3 is not None:
                    cv2.drawMarker(canvas, p3, team_col, cv2.MARKER_TILTED_CROSS, 20, 3)

        # 4c. Radios de presión del balón (3m, 5m, 10m)
        b_2d = None
        b_3d = None
        if ball_ok:
            bx_m, by_m = float(row["ball_x_m"]), float(row["ball_y_m"])
            b_2d = pitch_to_2d_px(bx_m, by_m)
            b_3d = pitch_to_video_px(bx_m, by_m, is_sh, H_mat)

            for r_m, r_col in [(10.0, (170, 210, 170)), (5.0, (140, 235, 235)), (3.0, (90, 255, 255))]:
                cv2.circle(canvas, b_2d, int(round(r_m * scale_2d)), r_col, 1, cv2.LINE_AA)
                poly_3d = [
                    pitch_to_video_px(bx_m + r_m * np.cos(th), by_m + r_m * np.sin(th), is_sh, H_mat)
                    for th in theta_circle
                ]
                poly_3d = [p for p in poly_3d if p is not None]
                if len(poly_3d) >= 6:
                    cv2.polylines(canvas, [np.array(poly_3d, dtype=np.int32)], True, r_col, 1, cv2.LINE_AA)

        near_L = int(row["nearest_slot_L"]) if ball_ok and np.isfinite(row["nearest_slot_L"]) else -1
        near_A = int(row["nearest_slot_A"]) if ball_ok and np.isfinite(row["nearest_slot_A"]) else -1

        # 4d. Jugadores
        for s in range(1, n_slots + 1):
            px_m, py_m = row[f"{s}_x_m"], row[f"{s}_y_m"]
            if not (np.isfinite(px_m) and np.isfinite(py_m)):
                continue

            p_col = col_L if slot_team[s - 1] == 0 else col_A
            pt_2d = pitch_to_2d_px(px_m, py_m)
            pt_3d = pitch_to_video_px(px_m, py_m, is_sh, H_mat)
            is_nearest = ball_ok and (s == near_L or s == near_A)

            vx, vy = row[f"{s}_vx"], row[f"{s}_vy"]
            if np.isfinite(vx) and np.isfinite(vy):
                cv2.arrowedLine(canvas, pt_2d, pitch_to_2d_px(px_m + 0.45 * vx, py_m + 0.45 * vy), (245, 245, 245), 2, tipLength=0.28)
                if pt_3d is not None:
                    pt_3d_v = pitch_to_video_px(px_m + 0.45 * vx, py_m + 0.45 * vy, is_sh, H_mat)
                    if pt_3d_v is not None:
                        cv2.arrowedLine(canvas, pt_3d, pt_3d_v, (245, 245, 245), 2, tipLength=0.28)

            if is_nearest:
                cv2.line(canvas, pt_2d, b_2d, p_col, 2, cv2.LINE_AA)
                cv2.circle(canvas, pt_2d, 13, (0, 255, 255), 2, cv2.LINE_AA)
                if pt_3d is not None and b_3d is not None:
                    cv2.line(canvas, pt_3d, b_3d, p_col, 2, cv2.LINE_AA)
                    cv2.ellipse(canvas, pt_3d, (18, 9), 0, 0, 360, (0, 255, 255), 2, cv2.LINE_AA)

            cv2.circle(canvas, pt_2d, 8, p_col, -1, cv2.LINE_AA)
            cv2.circle(canvas, pt_2d, 8, (255, 255, 255), 1, cv2.LINE_AA)
            cv2.putText(canvas, str(s), (pt_2d[0] - 7, pt_2d[1] + 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.34, (255, 255, 255), 1, cv2.LINE_AA)

            if pt_3d is not None:
                cv2.ellipse(canvas, pt_3d, (12, 6), 0, 0, 360, p_col, -1, cv2.LINE_AA)
                cv2.ellipse(canvas, pt_3d, (12, 6), 0, 0, 360, (255, 255, 255), 1, cv2.LINE_AA)
                cv2.putText(canvas, str(s), (pt_3d[0] - 7, pt_3d[1] - 9),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)

        # 4e. Balón
        if ball_ok:
            cv2.circle(canvas, b_2d, 6, (0, 230, 255), -1, cv2.LINE_AA)
            cv2.circle(canvas, b_2d, 6, (0, 0, 0), 2, cv2.LINE_AA)
            if b_3d is not None:
                cv2.circle(canvas, b_3d, 7, (0, 230, 255), -1, cv2.LINE_AA)
                cv2.circle(canvas, b_3d, 7, (0, 0, 0), 2, cv2.LINE_AA)

        # 4f. HUD Superior e Inferior
        cv2.rectangle(canvas, (10, 10), (540, 68), (20, 24, 28), -1)
        mins = int(row["time_s"] // 60)
        secs = row["time_s"] % 60.0
        cv2.putText(canvas, f"Seed={seed} | {row['period']} | vf={target_vf} | t={mins:02d}:{secs:05.2f}",
                    (20, 34), cv2.FONT_HERSHEY_SIMPLEX, 0.56, (240, 240, 240), 2, cv2.LINE_AA)

        st_txt = "BALON: OK (Paso 8+7)" if ball_ok else "BALON: AUSENTE -> FALLBACK (Paso 7)"
        st_col = (100, 235, 120) if ball_ok else (70, 175, 255)
        cv2.putText(canvas, st_txt, (20, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.50, st_col, 1, cv2.LINE_AA)

        if has_viterbi:
            st = str(row["pred_state"])
            st_map = {
                "L": ("VITERBI: LOCAL (L)", col_L),
                "A": ("VITERBI: AWAY (A)", col_A),
                "D": ("VITERBI: DEAD (D)", (200, 200, 200)),
                "X": ("VITERBI: UNKNOWN (X)", (0, 215, 255)),
            }
            lbl, lbl_col = st_map.get(st, (f"VITERBI: {st}", (255, 255, 255)))
            cv2.rectangle(canvas, (vid_w - 300, 10), (vid_w - 12, 52), (20, 24, 28), -1)
            cv2.putText(canvas, lbl, (vid_w - 288, 38), cv2.FONT_HERSHEY_SIMPLEX, 0.68, lbl_col, 2, cv2.LINE_AA)

        vis_txt = f"Vis L/A: {int(row['n_visible_L'])}/{int(row['n_visible_A'])} | VelMedia: {row['mean_speed_L']:.2f}/{row['mean_speed_A']:.2f} m/s | Parados: {row['frac_stationary_L']*100:.0f}%/{row['frac_stationary_A']*100:.0f}%"
        if ball_ok:
            ball_txt = f"d_min L/A: {row['d_min_L']:.2f}/{row['d_min_A']:.2f}m (diff={row['diff_d_min_LA']:+.2f}) | Close: {row['closing_speed_L']:+.1f}/{row['closing_speed_A']:+.1f} | R3m: {int(row['count_3m_L'])}/{int(row['count_3m_A'])}"
        else:
            ball_txt = "d_min L/A: NaN | CloseSpd: NaN | R3m: NaN"

        cv2.putText(canvas, vis_txt, (vid_w + margin_2d, canvas_h - 34),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (220, 225, 230), 1, cv2.LINE_AA)
        cv2.putText(canvas, ball_txt, (vid_w + margin_2d, canvas_h - 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (180, 230, 255), 1, cv2.LINE_AA)

        writer.write(canvas)

    cap.release()
    writer.release()
    print(f"Video Doble Vista exportado en: {output_path}")
    return output_path


# ==============================================================================
# EJECUCIÓN EN EL NOTEBOOK DE VITERBI
# ==============================================================================
slots_csv = PROCESSED_DIR / f"{games.GAME}_slot_teams_LA.csv"
slot_team = np.where(pd.read_csv(slots_csv)["team"] == "L", 0, 1) if slots_csv.exists() else D.load_slot_teams()

fh_mask = df_features["period"] == "first_half"
home_dir_fh = 1 if df_features.loc[fh_mask, "centroid_x_L"].median() <= df_features.loc[fh_mask, "centroid_x_A"].median() else -1

dual_video_path = export_dual_view_tactical_video(
    df_full=df_features,
    slot_team=slot_team,
    atd_summary={"home_first_half_dir": home_dir_fh},
    video_path=r"D:\Proyectos\Barcelona project local\data\tactical\BAR-ATH_tactical.mp4",
    seed=42,
    duration_s=20.0,
    df_viterbi=df_viterbi,
)

Video Doble Vista exportado en: D:\Proyectos\Barcelona project local\data\processed\videos\dual_view_seed42_first_half_vf25273_25772.mp4


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

# ==============================================================================
# 1. EVALUACIÓN CONTRA GROUND TRUTH (SI ESTÁ DISPONIBLE PARA ESTE PARTIDO)
# ==============================================================================
has_gt = False
if hasattr(D, "load_labels"):
    try:
        gt_raw = D.load_labels()
        if gt_raw is not None and len(gt_raw) > 0:
            # Detectar columna de frame y columna de etiqueta en gt_raw
            gt_df = gt_raw.copy()
            if "video_frame" not in gt_df.columns:
                if "frame_id" in gt_df.columns:
                    gt_df["video_frame"] = gt_df["frame_id"] - 1
                else:
                    gt_df["video_frame"] = np.arange(len(gt_df), dtype=np.int64)

            label_col = next((c for c in ["label", "state", "possession", "target", "y"] if c in gt_df.columns), None)
            if label_col is not None:
                gt_df["gt_state"] = gt_df[label_col].astype(str).str.strip().str.upper()
                eval_df = df_viterbi.merge(gt_df[["video_frame", "gt_state"]], on="video_frame", how="inner")
                eval_df = eval_df[eval_df["gt_state"].isin(["L", "A", "D"])].reset_index(drop=True)
                has_gt = len(eval_df) > 0
    except Exception as e:
        print(f"No se cargaron etiquetas GT (modo inferencia / test): {e}")

if has_gt:
    y_true = eval_df["gt_state"].to_numpy()
    y_pred = eval_df["pred_state"].to_numpy()

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, labels=["L", "A", "D"], average="macro")

    print("=== MÉTRICAS DE EVALUACIÓN VITERBI (TIEMPO DE JUEGO) ===")
    print(f"  Accuracy Global : {acc * 100:.2f}%")
    print(f"  Macro-F1        : {macro_f1 * 100:.2f}%\n")
    print(classification_report(y_true, y_pred, labels=["L", "A", "D"], digits=4))

    # Desglose: Con balón disponible vs Fallback colectivo (Paso 7)
    for avail_flag, desc in [(True, "Con Balón Disponible (Paso 8 + 7)"), (False, "Fallback Colectivo sin Balón (Solo Paso 7)")]:
        sub = eval_df[eval_df["ball_available"] == avail_flag]
        if len(sub) > 0:
            sub_acc = accuracy_score(sub["gt_state"], sub["pred_state"])
            sub_f1 = f1_score(sub["gt_state"], sub["pred_state"], labels=["L", "A", "D"], average="macro")
            print(f"  [{desc}] n={len(sub):6d} | Acc={sub_acc*100:.2f}% | Macro-F1={sub_f1*100:.2f}%")

    cm = confusion_matrix(y_true, y_pred, labels=["L", "A", "D"])
    print("\nMatriz de Confusión (filas=Real [L, A, D], cols=Pred [L, A, D]):")
    print(pd.DataFrame(cm, index=["Real_L", "Real_A", "Real_D"], columns=["Pred_L", "Pred_A", "Pred_D"]))
else:
    print("Modo inferencia sin etiquetas GT: revisa la distribución temporal en la gráfica inferior.")

# ==============================================================================
# 2. LÍNEA DE TIEMPO VISUAL (VENTANA DE 3 MINUTOS EN 1ª PARTE)
# ==============================================================================
state_to_num = {"L": 0, "A": 1, "D": 2}
cmap_lad = ListedColormap(["#1f77b4", "#d62728", "#7f7f7f"])  # Azul=L, Rojo=A, Gris=D

win_df = eval_df.iloc[:4500] if has_gt else df_viterbi.iloc[:4500]  # 4500 frames = 3 min @ 25fps
t_min = (win_df["video_frame"].to_numpy() - win_df["video_frame"].iloc[0]) / (25.0 * 60.0)

n_rows = 3 if has_gt else 2
fig, axes = plt.subplots(n_rows, 1, figsize=(15, 2.3 * n_rows), sharex=True)
plt.subplots_adjust(hspace=0.35)

# Fila 0: Predicción Viterbi
pred_num = win_df["pred_state"].map(state_to_num).to_numpy()[None, :]
axes[0].imshow(pred_num, aspect="auto", cmap=cmap_lad, vmin=0, vmax=2, extent=[t_min[0], t_min[-1], 0, 1])
axes[0].set_yticks([])
axes[0].set_title("Predicción Viterbi (Azul = Local 'L' | Rojo = Away 'A' | Gris = Dead 'D')")

row_idx = 1
if has_gt:
    gt_num = win_df["gt_state"].map(state_to_num).to_numpy()[None, :]
    axes[row_idx].imshow(gt_num, aspect="auto", cmap=cmap_lad, vmin=0, vmax=2, extent=[t_min[0], t_min[-1], 0, 1])
    axes[row_idx].set_yticks([])
    axes[row_idx].set_title("Ground Truth (Etiquetas Reales)")
    row_idx += 1

# Última fila: Disponibilidad del balón
ball_num = win_df["ball_available"].astype(int).to_numpy()[None, :]
axes[row_idx].imshow(ball_num, aspect="auto", cmap=ListedColormap(["#ff9800", "#2ca02c"]), vmin=0, vmax=1, extent=[t_min[0], t_min[-1], 0, 1])
axes[row_idx].set_yticks([])
axes[row_idx].set_title("Disponibilidad del Balón (Verde = Paso 8 Activo | Naranja = Fallback Colectivo Paso 7)")
axes[row_idx].set_xlabel("Minutos desde el inicio de la ventana (1ª parte)")

plt.show()

# ==============================================================================
# 3. RECONSTRUCCIÓN REJILLA COMPLETA (1..N_FRAMES) Y EXPORTACIÓN FINAL
# ==============================================================================
n_total = D.N_FRAMES
full_sub = pd.DataFrame({
    "frame_id": np.arange(1, n_total + 1, dtype=np.int64),
    "video_frame": np.arange(0, n_total, dtype=np.int64),
})

full_sub = full_sub.merge(
    df_viterbi[["video_frame", "period", "pred_state", "ball_available", "use_team_fallback"]],
    on="video_frame",
    how="left",
)
# Fuera de first_half / second_half (antes del saque, descanso, post-partido) -> 'D'
full_sub["period"] = full_sub["period"].fillna("non_playing")
full_sub["state"] = full_sub["pred_state"].fillna("D")

SUB_DIR = Path(os.environ["MTAH_DATA_DIR"]).parent / "submissions"
SUB_DIR.mkdir(parents=True, exist_ok=True)

SUB_PATH = SUB_DIR / f"{games.GAME}_viterbi_LA_submission.csv"
full_sub[["frame_id", "video_frame", "state"]].to_csv(SUB_PATH, index=False)

print(f"\nArchivo final exportado en: {SUB_PATH}")
print("Distribución global en rejilla completa (1..N_FRAMES):")
print((full_sub["state"].value_counts(normalize=True) * 100.0).round(2).to_string())